# Inspecting a Generated World

This notebook walks you through generating a synthetic CRM dataset with **leadforge** and exploring what's inside the output bundle.

**Prerequisites:** `pip install -e ".[dev]"` from the repo root.

We'll cover:
1. Generating a bundle via the Python API
2. Exploring `manifest.json` — provenance, row counts, file hashes
3. Loading the relational tables and examining FK relationships
4. Inspecting the task splits (train/valid/test)
5. Reading the dataset card and feature dictionary

## 1. Generate a bundle

We use `Generator.from_recipe()` to create a small world (500 leads) in `student_public` mode with `intro` difficulty. The bundle is written to a temporary directory so nothing lingers after the notebook.

In [ ]:
import tempfile
from pathlib import Path

from leadforge.api import Generator

tmpdir = tempfile.mkdtemp(prefix="leadforge_demo_")
bundle_path = Path(tmpdir) / "demo_bundle"

gen = Generator.from_recipe(
    "b2b_saas_procurement_v1",
    seed=42,
    exposure_mode="student_public",
    difficulty="intro",
)
bundle = gen.generate(n_leads=500)
bundle.save(str(bundle_path))

print(f"Bundle written to: {bundle_path}")

Let's see what files were created:

In [ ]:
for p in sorted(bundle_path.rglob("*")):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f"  {p.relative_to(bundle_path)}  ({size_kb:.1f} KB)")

## 2. Explore the manifest

`manifest.json` is the bundle's provenance record. It captures the recipe, seed, package version, exposure mode, row counts, and SHA-256 hashes for every data file — everything you need to reproduce or verify the dataset.

In [ ]:
import json

with open(bundle_path / "manifest.json") as f:
    manifest = json.load(f)

# Top-level provenance fields
for key in [
    "package_version",
    "recipe_id",
    "seed",
    "exposure_mode",
    "difficulty",
    "generation_timestamp",
    "bundle_schema_version",
]:
    print(f"{key}: {manifest.get(key)}")

In [ ]:
# Table inventory: row counts and file hashes
print("\nRelational tables:")
for name, info in manifest["tables"].items():
    print(f"  {name:20s}  {info['row_count']:>6,} rows  sha256={info['sha256'][:12]}...")

print("\nTask splits:")
for task_id, splits in manifest["tasks"].items():
    print(f"  {task_id}:")
    for split_name, info in splits.items():
        print(f"    {split_name:6s}  {info['row_count']:>5,} rows")

## 3. Relational tables

The bundle contains 9 relational tables stored as Parquet files under `tables/`. These represent the full CRM world: accounts, contacts, leads, their interactions (touches, sessions, sales activities), and outcomes (opportunities, customers, subscriptions).

In [ ]:
import pandas as pd

tables = {}
for parquet_file in sorted((bundle_path / "tables").glob("*.parquet")):
    name = parquet_file.stem
    tables[name] = pd.read_parquet(parquet_file)

# Summary of all tables
summary = pd.DataFrame(
    [{"table": name, "rows": len(df), "columns": len(df.columns)} for name, df in tables.items()]
)
print(summary.to_string(index=False))

In [ ]:
# Sample rows from the leads table
tables["leads"].head(3)

In [ ]:
# Sample rows from the touches table
tables["touches"].head(3)

### FK relationships

The tables are linked by foreign keys (e.g., every lead references an account and a contact). Let's verify one relationship and see how the tables connect.

In [ ]:
# Every lead's account_id should exist in the accounts table
lead_account_ids = set(tables["leads"]["account_id"])
account_ids = set(tables["accounts"]["account_id"])
orphans = lead_account_ids - account_ids
print(f"FK check: {len(orphans)} orphan account_ids (expect 0)")

print(f"Accounts: {len(account_ids)}")
print(f"Contacts: {len(tables['contacts'])}")
print(f"Leads: {len(tables['leads'])}")
print(f"Leads per account (mean): {len(tables['leads']) / len(account_ids):.1f}")
print(f"Touches per lead (mean): {len(tables['touches']) / len(tables['leads']):.1f}")

## 4. Task splits

The primary task (`converted_within_90_days`) is exported as train/valid/test Parquet splits under `tasks/`. Each row is a lead snapshot — a flat, ML-ready feature vector anchored at the snapshot date. No post-snapshot data leaks into these features.

In [ ]:
task_dir = bundle_path / "tasks" / "converted_within_90_days"

splits = {}
for split_file in sorted(task_dir.glob("*.parquet")):
    splits[split_file.stem] = pd.read_parquet(split_file)

for name, df in splits.items():
    n_pos = df["converted_within_90_days"].sum()
    rate = n_pos / len(df) * 100
    print(f"{name:6s}: {len(df):>4} rows, {n_pos:>3} converted ({rate:.1f}%)")

In [ ]:
# Feature overview from the train split
train = splits["train"]
print(f"Features: {len(train.columns)} columns\n")
train.dtypes

In [ ]:
# Quick summary statistics for numeric features
train.describe().T

### Task manifest

`task_manifest.json` records the split ratios and label column for reproducibility.

In [ ]:
with open(task_dir / "task_manifest.json") as f:
    task_manifest = json.load(f)

task_manifest

## 5. Dataset card and feature dictionary

Every bundle includes a human-readable dataset card (Markdown) and a machine-readable feature dictionary (CSV) describing each column in the task table.

In [ ]:
# Dataset card (first 40 lines)
card_text = (bundle_path / "dataset_card.md").read_text()
print("\n".join(card_text.splitlines()[:40]))
print(f"\n... ({len(card_text.splitlines())} lines total)")

In [ ]:
# Feature dictionary
feat_dict = pd.read_csv(bundle_path / "feature_dictionary.csv")
feat_dict

## What's next?

This bundle was generated in **`student_public`** mode, which excludes the hidden causal structure behind the data. leadforge also supports a **`research_instructor`** mode that includes the full world graph, latent variable registry, and mechanism summaries — useful for teaching causal inference or evaluating model interpretability. That's a topic for a future notebook.

For now, you have everything you need to start building models on the task splits!

## Cleanup

In [ ]:
import shutil

shutil.rmtree(tmpdir)
print(f"Cleaned up {tmpdir}")